# Experiment 42 — Three-factor backward-free SparseWalker

A surgical test on top of the successful Experiment 39 local-contrastive learner.

**Unchanged:** corrected SparseWalker v1.1, random fixed graph, 32 random negatives, analytic local item contrastive learning, competitive router/key learning, no warm start, no optimizer, no backward/autograd.

**Changed:** concept values, graph context projection, and message projection use a shared three-factor rule: local pre activity × local target/post activity × centered prediction reward. A 4-event eligibility trace (decay 0.6) provides short delayed credit.

Reference validation NDCG@10: LC-v1 **0.040516**, SASRec **0.042968**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, runpy, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments',f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run / resume

The default experiment uses reward baseline EMA β=0.99, modulator gain 4.0, trace length 4, and trace decay 0.6. Set `RESUME=True` after a runtime crash.


In [ ]:
RESUME=False
SCRIPT=f'{REPO}/experiments/run_amazon_three_factor_walker.py'
argv=[SCRIPT,'--dataset','beauty','--epochs','70','--batch-size','512','--eval-batch-size','1024','--reward-beta','0.99','--mod-gain','4.0','--trace-len','4','--trace-decay','0.6']
if RESUME: argv.append('--resume')
sys.argv=argv
runpy.run_path(SCRIPT,run_name='__main__')


## Inspect trajectory

The key question is whether a shared prediction-error modulator improves the successful local learner. Watch both ranking quality and whether the modulator remains balanced rather than saturating.


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_three_factor/beauty/seed42')
hp=root/'history.json'
if hp.exists():
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','mean_contrastive_margin','mean_reward','reward_baseline','mean_abs_prediction_error','mean_abs_modulator','positive_modulator_fraction','negative_modulator_fraction','eligibility_delayed_credit_fraction','mean_three_factor_value_change','mean_three_factor_context_update','mean_three_factor_message_update','val_NDCG@10','val_HR@10','positions_per_s']
    display(h[[c for c in cols if c in h.columns]])
    if len(h):
        best=h.loc[h['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
        print('LC_V1_BEST_VAL_NDCG',0.040515991131704246)
        print('SASREC_VAL_NDCG',0.04296780165590764)
else:
    print('No history yet.')


## Crash recovery / final result

`last.pt` is written every epoch. If training already reached epoch 70, rerunning with `RESUME=True` loads the saved state and emits the final frozen-test result.


In [ ]:
rp=root/'result.json'
bp=root/'best.pt'
lp=root/'last.pt'
print('best.pt',bp.exists(),'last.pt',lp.exists(),'result.json',rp.exists())
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()),indent=2))
elif lp.exists():
    ck=torch.load(lp,map_location='cpu')
    print('RECOVERABLE_FROM_EPOCH',ck['epoch'],'BEST_EPOCH',ck.get('best_epoch'),'BEST_VAL',ck.get('best'))
